# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [1]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [3]:
import logging

logging.basicConfig(
    level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger("single_agent")

In [8]:
# 🤖 AGENT FUNCTION (TO IMPLEMENT)
import re

def agent(query: str) -> dict:
    if not isinstance(query, str) or not query.strip():
        logger.warning("Empty or invalid query received.")
        return {"type": "error", "result": "Empty or invalid query."}
    query_lower = query.lower()
    logger.info(f"Received query: {query!r}")

    try:
        # Route 1: Calculation
        if "calculate" in query_lower:
            # Pull out the numeric/operator portion of the query so the
            # word "calculate" itself never reaches eval().
            match = re.search(r"[\d\.\s\+\-\*\/\(\)\%]+", query)
            expression = match.group().strip() if match else ""

            if not expression:
                logger.error("No valid mathematical expression found in query.")
                return {"type": "error", "result": "No valid mathematical expression found."}

            result = calculator(expression)
            if result == "Error in calculation":
                logger.error(f"Calculator failed on expression: {expression!r}")
                return {"type": "error", "result": f"Could not evaluate expression: '{expression}'"}

            logger.info(f"Calculation successful: {expression} = {result}")
            return {"type": "calculation", "result": result}

        #  Route 2: Keyword extraction
        elif "keywords" in query_lower:
            # Strip common trigger phrases so only the target text is sent
            # to the keyword tool.
            text = re.sub(r"extract\s+keywords\s+from", "", query, flags=re.IGNORECASE).strip()
            text = re.sub(r"keywords", "", text, flags=re.IGNORECASE).strip()
            if not text:
                text = query

            keywords = extract_keywords(text)
            logger.info(f"Extracted keywords: {keywords}")
            return {"type": "keywords", "result": keywords}

        # Route 3: General fallback
        else:
            logger.info("Routed to general fallback handler.")
            return {
                "type": "general",
                "result": (
                    f"I received your message: '{query.strip()}'. "
                    "I can help with calculations (say 'calculate ...') "
                    "or keyword extraction (say 'extract keywords from ...')."
                )
            }

    except Exception as e:
        logger.exception("Unexpected error in agent routing.")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}


## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [11]:
# 🧪 Test Cases
import json

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate 10 / 0",          # error case: bad math
    "keywords",                  # edge case: trigger word with no text
]

for q in queries:
    print("Query:   ", q)
    response = agent(q)
    print("Response:", json.dumps(response, indent=2))
    print("-" * 50)


ERROR:single_agent:Calculator failed on expression: '10 / 0'


Query:    Calculate 20 + 5
Response: {
  "type": "calculation",
  "result": "25"
}
--------------------------------------------------
Query:    Extract keywords from Artificial Intelligence is transforming industries
Response: {
  "type": "keywords",
  "result": [
    "industries",
    "intelligence",
    "artificial",
    "transforming"
  ]
}
--------------------------------------------------
Query:    What is machine learning?
Response: {
  "type": "general",
  "result": "I received your message: 'What is machine learning?'. I can help with calculations (say 'calculate ...') or keyword extraction (say 'extract keywords from ...')."
}
--------------------------------------------------
Query:    Calculate 10 / 0
Response: {
  "type": "error",
  "result": "Could not evaluate expression: '10 / 0'"
}
--------------------------------------------------
Query:    keywords
Response: {
  "type": "keywords",
  "result": [
    "keywords"
  ]
}
--------------------------------------------------


In [13]:
import json

while True:
    try:
        user_input = input("Enter query (type 'exit' to stop): ")
    except (EOFError, KeyboardInterrupt):
        print("\nExiting interactive mode.")
        break

    if user_input.strip().lower() == "exit":
        break

    response = agent(user_input)
    print("Response:", json.dumps(response, indent=2))


Enter query (type 'exit' to stop): exit


The section above implements the agent as a single python function with if /elif /else routing. Now we do see the exact same logic by using the langgraph , so the routing , tool calls and state are all first-class graph nodes/edges insted of being hidden inside one function body.

In [14]:
%pip install -q langgraph

In [17]:


from typing import TypedDict, Optional, Any
from langgraph.graph import StateGraph, END
import re
import logging


logger = logging.getLogger("single_agent")

class AgentState(TypedDict):
    query: str
    type: Optional[str]
    result: Optional[Any]


def router_node(state: AgentState) -> AgentState:
    logger.info(f"[LangGraph] Routing query: {state['query']!r}")
    return state


def route_decision(state: AgentState) -> str:
    q = state["query"].lower()
    if "calculate" in q:
        return "calculator_node"
    elif "keywords" in q:
        return "keyword_node"
    else:
        return "general_node"


def calculator_node(state: AgentState) -> AgentState:
    query = state["query"]
    match = re.search(r"[\d\.\s\+\-\*\/\(\)\%]+", query)
    expression = match.group().strip() if match else ""

    if not expression:
        return {**state, "type": "error", "result": "No valid mathematical expression found."}

    result = calculator(expression)
    if result == "Error in calculation":
        return {**state, "type": "error", "result": f"Could not evaluate expression: '{expression}'"}

    return {**state, "type": "calculation", "result": result}


def keyword_node(state: AgentState) -> AgentState:
    query = state["query"]
    text = re.sub(r"extract\s+keywords\s+from", "", query, flags=re.IGNORECASE).strip()
    text = re.sub(r"keywords", "", text, flags=re.IGNORECASE).strip()
    if not text:
        text = query
    keywords = extract_keywords(text)
    return {**state, "type": "keywords", "result": keywords}


def general_node(state: AgentState) -> AgentState:
    query = state["query"]
    result = (
        f"I received your message: '{query.strip()}'. "
        "I can help with calculations (say 'calculate ...') "
        "or keyword extraction (say 'extract keywords from ...')."
    )
    return {**state, "type": "general", "result": result}


graph = StateGraph(AgentState)
graph.add_node("router", router_node)
graph.add_node("calculator_node", calculator_node)
graph.add_node("keyword_node", keyword_node)
graph.add_node("general_node", general_node)

graph.set_entry_point("router")
graph.add_conditional_edges(
    "router",
    route_decision,
    {
        "calculator_node": "calculator_node",
        "keyword_node": "keyword_node",
        "general_node": "general_node",
    },
)
graph.add_edge("calculator_node", END)
graph.add_edge("keyword_node", END)
graph.add_edge("general_node", END)

langgraph_agent = graph.compile()
print("LangGraph agent compiled successfully.")


LangGraph agent compiled successfully.


In [19]:


import json

def run_langgraph_agent(query: str) -> dict:
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Empty or invalid query."}
    try:
        state = langgraph_agent.invoke({"query": query, "type": None, "result": None})
        return {"type": state["type"], "result": state["result"]}
    except Exception as e:
        logger.exception("LangGraph agent failed.")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}


for q in queries:
    print("Query:   ", q)
    response = run_langgraph_agent(q)
    print("Response:", json.dumps(response, indent=2))
    print("-" * 50)


Query:    Calculate 20 + 5
Response: {
  "type": "calculation",
  "result": "25"
}
--------------------------------------------------
Query:    Extract keywords from Artificial Intelligence is transforming industries
Response: {
  "type": "keywords",
  "result": [
    "industries",
    "intelligence",
    "artificial",
    "transforming"
  ]
}
--------------------------------------------------
Query:    What is machine learning?
Response: {
  "type": "general",
  "result": "I received your message: 'What is machine learning?'. I can help with calculations (say 'calculate ...') or keyword extraction (say 'extract keywords from ...')."
}
--------------------------------------------------
Query:    Calculate 10 / 0
Response: {
  "type": "error",
  "result": "Could not evaluate expression: '10 / 0'"
}
--------------------------------------------------
Query:    keywords
Response: {
  "type": "keywords",
  "result": [
    "keywords"
  ]
}
--------------------------------------------------
